# Git Explainer Eval Workbench

This notebook gives you an interactive place to:

- run the agent on a single query against the local project
- run an offline-friendly smoke version of the benchmark or the full suite
- save notebook results to `eval/results.notebook.json`
- plot pass/fail counts, retrieval accuracy, citation metrics, faithfulness, and latency


In [ ]:
from __future__ import annotations

import sys
import time
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import JSON, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "git_explainer").exists() and (candidate / "eval").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EVAL_DIR = PROJECT_ROOT / "eval"
RESULTS_PATH = EVAL_DIR / "results.notebook.json"

from eval.evaluate import load_benchmark, run_case, save_results, setup_repos, summarize_scores
from git_explainer.orchestrator import explain_code_history

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

print(f"Project root: {PROJECT_ROOT}")
print(f"Eval directory: {EVAL_DIR}")
print(f"Notebook results path: {RESULTS_PATH}")


## Notebook Configuration

The defaults below are chosen so the notebook can exercise the local repo without requiring LLM or GitHub access. Flip `RUN_FULL_BENCHMARK` and `OFFLINE_MODE` when you want the complete PRD-style run.


In [ ]:
USE_LLM = False

AGENT_QUERY = {
    "file_path": "git_explainer/orchestrator.py",
    "question": "Why does the agent fall back to a deterministic summary?",
    "owner": None,
    "repo_name": None,
}

RUN_FULL_BENCHMARK = False
OFFLINE_MODE = True

SELECTED_IDS = [
    "file-reader-revision-fix",
    "line-range-slice-fix",
    "question-issue-lookup-library",
    "adversarial-end-before-start",
]
SELECTED_TAGS = None

# For the full benchmark, switch to:
# RUN_FULL_BENCHMARK = True
# OFFLINE_MODE = False
# SELECTED_IDS = None


In [ ]:
def selected_cases(all_cases):
    cases = list(all_cases)
    if RUN_FULL_BENCHMARK:
        return cases
    if SELECTED_IDS:
        selected = set(SELECTED_IDS)
        return [case for case in cases if case.id in selected]
    if SELECTED_TAGS:
        selected_tags = set(SELECTED_TAGS)
        return [case for case in cases if selected_tags.intersection(case.tags)]
    return cases


def prepare_cases_for_notebook(all_cases):
    prepared = []
    skipped = []
    for case in selected_cases(all_cases):
        candidate = deepcopy(case)
        if OFFLINE_MODE and (candidate.expected.get("pr_numbers") or candidate.expected.get("issue_numbers")):
            skipped.append((candidate.id, "requires GitHub metadata for its checks"))
            continue
        if OFFLINE_MODE and candidate.repo_name == PROJECT_ROOT.name:
            candidate.repo_url = str(PROJECT_ROOT)
            candidate.owner = None
            candidate.repo_name = None
        prepared.append(candidate)
    return prepared, skipped


def case_scores_frame(scores):
    rows = []
    for score in scores:
        faithfulness = score.metrics.get("faithfulness_rubric", {})
        rows.append(
            {
                "case_id": score.case_id,
                "status": "error" if score.error else ("passed" if score.passed else "failed"),
                "elapsed_seconds": score.elapsed_seconds,
                "retrieval_accuracy": score.metrics.get("retrieval_accuracy"),
                "citation_coverage": score.metrics.get("citation_coverage"),
                "citation_validity": score.metrics.get("citation_validity"),
                "faithfulness_overall": faithfulness.get("overall"),
                "checks": ", ".join(name for name, ok in score.checks.items() if not ok),
                "error": score.error,
            }
        )
    if not rows:
        return pd.DataFrame(
            columns=[
                "case_id",
                "status",
                "elapsed_seconds",
                "retrieval_accuracy",
                "citation_coverage",
                "citation_validity",
                "faithfulness_overall",
                "checks",
                "error",
            ]
        )
    frame = pd.DataFrame(rows)
    return frame.sort_values(["status", "case_id"]).reset_index(drop=True)


def summary_frame(summary):
    rows = [
        {"metric": "Pass rate", "value": summary.get("pass_rate")},
        {"metric": "Retrieval accuracy", "value": summary["retrieval"].get("accuracy")},
        {"metric": "Citation coverage", "value": summary["citation"].get("coverage")},
        {"metric": "Citation validity", "value": summary["citation"].get("validity")},
        {
            "metric": "Faithfulness (/5)",
            "value": summary["faithfulness_rubric"].get("average"),
        },
        {"metric": "Avg latency (s)", "value": summary["latency"].get("average_seconds")},
        {"metric": "P95 latency (s)", "value": summary["latency"].get("p95_seconds")},
    ]
    return pd.DataFrame(rows)


## Single Agent Query

This is a quick sanity-check cell for the agent itself before running the benchmark.


In [ ]:
single_result = explain_code_history(
    repo_path=str(PROJECT_ROOT),
    file_path=AGENT_QUERY["file_path"],
    question=AGENT_QUERY["question"],
    owner=AGENT_QUERY["owner"],
    repo_name=AGENT_QUERY["repo_name"],
    use_llm=USE_LLM,
)

display(JSON(single_result, expanded=False))


## Benchmark Run

The default configuration uses a smoke subset that stays local and deterministic. When you are ready for the full suite, turn off offline mode and flip `RUN_FULL_BENCHMARK` in the config cell.


In [ ]:
benchmark_cases = load_benchmark(EVAL_DIR / "benchmark.json")
cases_to_run, skipped_cases = prepare_cases_for_notebook(benchmark_cases)

print(f"Loaded {len(benchmark_cases)} benchmark cases.")
if skipped_cases:
    print("Skipped in this notebook configuration:")
    for case_id, reason in skipped_cases:
        print(f"- {case_id}: {reason}")

if not cases_to_run:
    raise RuntimeError("No benchmark cases remain after applying the notebook filters.")

repo_map = setup_repos(cases_to_run)
scores = []
benchmark_start = time.monotonic()

for index, case in enumerate(cases_to_run, start=1):
    print(f"[{index}/{len(cases_to_run)}] Running {case.id}...")
    scores.append(run_case(case, repo_map.get(case.repo_url, ""), no_llm=not USE_LLM))

benchmark_elapsed = time.monotonic() - benchmark_start
summary = summarize_scores(cases_to_run, scores, benchmark_elapsed)
save_results(scores, summary, RESULTS_PATH)

print(f"Saved notebook results to {RESULTS_PATH}")
display(JSON(summary, expanded=True))


In [ ]:
case_df = case_scores_frame(scores)
summary_df = summary_frame(summary)

display(summary_df)
display(case_df)


## Plots

These plots are driven directly from the results produced in the benchmark cell above.


In [ ]:
if case_df.empty:
    raise RuntimeError("Run the benchmark cell before plotting results.")

plot_df = case_df.copy()
for column in ["retrieval_accuracy", "citation_coverage", "citation_validity"]:
    plot_df[column] = plot_df[column].fillna(0.0)
plot_df["faithfulness_overall"] = plot_df["faithfulness_overall"].fillna(0.0)

status_colors = {"passed": "#2b8a3e", "failed": "#c92a2a", "error": "#f08c00"}
aggregate_metrics = pd.Series(
    {
        "Pass rate": summary.get("pass_rate") or 0.0,
        "Retrieval": summary["retrieval"].get("accuracy") or 0.0,
        "Citation cov.": summary["citation"].get("coverage") or 0.0,
        "Citation valid.": summary["citation"].get("validity") or 0.0,
        "Faithfulness": (summary["faithfulness_rubric"].get("average") or 0.0) / 5.0,
    }
)

fig, axes = plt.subplots(2, 2, figsize=(18, 11))

count_series = pd.Series(summary["counts"])[["passed", "failed", "errors"]]
count_series.plot(
    kind="bar",
    ax=axes[0, 0],
    color=[status_colors["passed"], status_colors["failed"], status_colors["error"]],
    rot=0,
)
axes[0, 0].set_title("Benchmark outcomes")
axes[0, 0].set_ylabel("Case count")

aggregate_metrics.plot(kind="bar", ax=axes[0, 1], color="#1c7ed6", rot=20)
axes[0, 1].set_title("Aggregate benchmark metrics")
axes[0, 1].set_ylabel("Normalized score")
axes[0, 1].set_ylim(0, 1.05)

metric_columns = ["retrieval_accuracy", "citation_coverage", "citation_validity"]
plot_df.set_index("case_id")[metric_columns].plot(kind="bar", ax=axes[1, 0], rot=45)
axes[1, 0].set_title("Per-case evidence metrics")
axes[1, 0].set_ylabel("Score")
axes[1, 0].set_ylim(0, 1.05)

axes[1, 1].scatter(
    plot_df["elapsed_seconds"],
    plot_df["faithfulness_overall"],
    c=plot_df["status"].map(status_colors),
    s=90,
)
for _, row in plot_df.iterrows():
    axes[1, 1].annotate(
        row["case_id"],
        (row["elapsed_seconds"], row["faithfulness_overall"]),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points",
    )
axes[1, 1].set_title("Latency vs. faithfulness")
axes[1, 1].set_xlabel("Elapsed seconds")
axes[1, 1].set_ylabel("Faithfulness (/5)")
axes[1, 1].set_ylim(0, 5.2)

plt.tight_layout()
plt.show()
